In [1]:
# ============================================================================
# MARKETLAB DAY 2 - SETUP & PACKAGE INSTALLATION
# ============================================================================

print("🔥 MARKETLAB DAY 2 - DEEP LEARNING SETUP")
print("="*80)

# Mount Google Drive
print("\n📁 Mounting Google Drive...")
from google.colab import drive
drive.mount('/content/drive')
print("✅ Drive mounted successfully!")

# Check TensorFlow version (already installed in Colab)
print("\n📦 Checking TensorFlow installation...")
import tensorflow as tf
print(f"   ✅ TensorFlow version: {tf.__version__}")
print(f"   ✅ GPU available: {tf.config.list_physical_devices('GPU')}")

# Import required libraries
print("\n📚 Importing libraries...")
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import json
from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print("✅ All libraries imported!")

print("\n" + "="*80)
print("✅ SETUP COMPLETE!")
print("✅ Ready to run CELL 2 for training!")
print("="*80)

🔥 MARKETLAB DAY 2 - DEEP LEARNING SETUP

📁 Mounting Google Drive...
Mounted at /content/drive
✅ Drive mounted successfully!

📦 Checking TensorFlow installation...
   ✅ TensorFlow version: 2.19.0
   ✅ GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

📚 Importing libraries...
✅ All libraries imported!

✅ SETUP COMPLETE!
✅ Ready to run CELL 2 for training!


In [1]:
# ============================================================================
# MARKETLAB DAY 2 - DEEP LEARNING TRAINING
# ============================================================================

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.models import Sequential
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import json
from pathlib import Path
from datetime import datetime
import time
import warnings
warnings.filterwarnings('ignore')
import gc

print("🔥 MARKETLAB DAY 2 - DEEP LEARNING")
print("="*80)
print(f"TensorFlow: {tf.__version__}")
print(f"GPU: {tf.config.list_physical_devices('GPU')}")
print("="*80 + "\n")

# ============================================================================
# CONFIGURATION
# ============================================================================

STOCKS = [
    'RELIANCE.NS', 'TCS.NS', 'HDFCBANK.NS', 'INFY.NS', 'TATAMOTORS.NS',
    'BAJFINANCE.NS', 'ASIANPAINT.NS', 'SUNPHARMA.NS', 'ITC.NS', 'COALINDIA.NS'
]

LOOKBACK = 60  # 60-day sequences
EPOCHS = 50
BATCH_SIZE = 32
LEARNING_RATE = 0.001

# ============================================================================
# DATA LOADING
# ============================================================================

def load_csv(ticker):
    """Load stock data from CSV"""
    path = f"/content/drive/MyDrive/MarketLab_BEAST/stock_data/{ticker}.csv"
    with open(path, 'r') as f:
        first_line = f.readline()

    if 'Price","Open' in first_line:
        df = pd.read_csv(path)
        df['Date'] = pd.to_datetime(df['Date'], dayfirst=True)
        df = df.set_index('Date')
        df = df.rename(columns={'Price': 'Close', 'Vol.': 'Volume'})
        if df['Volume'].dtype == 'object':
            df['Volume'] = df['Volume'].str.replace('M', '').str.replace('K', '').str.replace(',', '')
            df['Volume'] = df['Volume'].apply(lambda x: float(x)*1000000 if 'M' in str(x) else (float(x)*1000 if 'K' in str(x) else float(x) if x else 0))
        df = df[['Open', 'High', 'Low', 'Close', 'Volume']]
    else:
        df = pd.read_csv(path, skiprows=[1])
        df = df.iloc[1:]
        df['Date'] = pd.to_datetime(df['Price'])
        df = df.set_index('Date')
        df = df[['Open', 'High', 'Low', 'Close', 'Volume']]

    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    return df.dropna().sort_index()

# ============================================================================
# FEATURE ENGINEERING (325 FEATURES - SAME AS DAY 1)
# ============================================================================

def generate_325_features(df):
    """Generate all 325 technical features"""

    # Price features
    df['hl_ratio'] = df['High'] / df['Low']
    df['co_ratio'] = df['Close'] / df['Open']
    df['oc_diff'] = df['Open'] - df['Close']
    df['hl_diff'] = df['High'] - df['Low']
    df['body_size'] = abs(df['Close'] - df['Open'])
    df['upper_shadow'] = df['High'] - df[['Open','Close']].max(axis=1)
    df['lower_shadow'] = df[['Open','Close']].min(axis=1) - df['Low']
    df['body_position'] = (df['Close'] - df['Low']) / (df['High'] - df['Low'])

    # Returns
    for p in [1,2,3,5,7,10,14,21,30,60,90,120,180,252]:
        df[f'ret_{p}'] = df['Close'].pct_change(p)
        df[f'log_ret_{p}'] = np.log(df['Close'] / df['Close'].shift(p))

    # Moving averages
    for p in [3,5,7,10,14,20,30,50,100,150,200]:
        df[f'sma_{p}'] = df['Close'].rolling(p).mean()
        df[f'ema_{p}'] = df['Close'].ewm(span=p).mean()
        df[f'close_sma_ratio_{p}'] = df['Close'] / df[f'sma_{p}']
        df[f'close_ema_ratio_{p}'] = df['Close'] / df[f'ema_{p}']

    # Volatility
    for p in [5,10,20,30,60,90]:
        df[f'volatility_{p}'] = df['Close'].pct_change().rolling(p).std()
        df[f'volatility_{p}_annual'] = df[f'volatility_{p}'] * np.sqrt(252)
        df[f'parkinson_{p}'] = np.sqrt((1/(4*np.log(2))) * ((np.log(df['High']/df['Low']))**2).rolling(p).mean())
        df[f'garman_klass_{p}'] = np.sqrt(0.5 * ((np.log(df['High']/df['Low']))**2).rolling(p).mean() - (2*np.log(2)-1) * ((np.log(df['Close']/df['Open']))**2).rolling(p).mean())

    # RSI
    for p in [7,9,14,21,28]:
        delta = df['Close'].diff()
        gain = delta.where(delta > 0, 0).rolling(p).mean()
        loss = -delta.where(delta < 0, 0).rolling(p).mean()
        rs = gain / loss
        df[f'rsi_{p}'] = 100 - (100 / (1 + rs))
        df[f'rsi_{p}_slope'] = df[f'rsi_{p}'].diff(5)
        df[f'rsi_{p}_ema'] = df[f'rsi_{p}'].ewm(span=5).mean()

    # MACD
    for fast, slow, signal in [(12,26,9), (5,35,5), (19,39,9)]:
        ema_fast = df['Close'].ewm(span=fast).mean()
        ema_slow = df['Close'].ewm(span=slow).mean()
        df[f'macd_{fast}_{slow}'] = ema_fast - ema_slow
        df[f'macd_signal_{fast}_{slow}_{signal}'] = df[f'macd_{fast}_{slow}'].ewm(span=signal).mean()
        df[f'macd_hist_{fast}_{slow}_{signal}'] = df[f'macd_{fast}_{slow}'] - df[f'macd_signal_{fast}_{slow}_{signal}']
        df[f'macd_slope_{fast}_{slow}'] = df[f'macd_{fast}_{slow}'].diff(3)

    # Bollinger Bands
    for p in [10,20,30,50]:
        for std_mult in [1.5, 2.0, 2.5]:
            sma = df['Close'].rolling(p).mean()
            std = df['Close'].rolling(p).std()
            df[f'bb_upper_{p}_{std_mult}'] = sma + (std_mult * std)
            df[f'bb_lower_{p}_{std_mult}'] = sma - (std_mult * std)
            df[f'bb_width_{p}_{std_mult}'] = (df[f'bb_upper_{p}_{std_mult}'] - df[f'bb_lower_{p}_{std_mult}']) / sma
            df[f'bb_position_{p}_{std_mult}'] = (df['Close'] - df[f'bb_lower_{p}_{std_mult}']) / (df[f'bb_upper_{p}_{std_mult}'] - df[f'bb_lower_{p}_{std_mult}'])

    # Volume
    for p in [5,10,20,30,50]:
        df[f'volume_sma_{p}'] = df['Volume'].rolling(p).mean()
        df[f'volume_ratio_{p}'] = df['Volume'] / df[f'volume_sma_{p}']
        df[f'volume_std_{p}'] = df['Volume'].rolling(p).std()
        df[f'volume_cv_{p}'] = df[f'volume_std_{p}'] / df[f'volume_sma_{p}']
        df[f'obv_{p}'] = (np.sign(df['Close'].diff()) * df['Volume']).rolling(p).sum()
        df[f'vwap_{p}'] = (df['Close'] * df['Volume']).rolling(p).sum() / df['Volume'].rolling(p).sum()

    # Momentum
    for p in [5,10,20,30,50]:
        df[f'momentum_{p}'] = df['Close'] - df['Close'].shift(p)
        df[f'roc_{p}'] = df['Close'].pct_change(p) * 100
        df[f'trix_{p}'] = df['Close'].ewm(span=p).mean().ewm(span=p).mean().ewm(span=p).mean().pct_change()

    # ATR
    for p in [7,14,21,30]:
        high_low = df['High'] - df['Low']
        high_close = np.abs(df['High'] - df['Close'].shift())
        low_close = np.abs(df['Low'] - df['Close'].shift())
        ranges = pd.concat([high_low, high_close, low_close], axis=1)
        true_range = ranges.max(axis=1)
        df[f'atr_{p}'] = true_range.rolling(p).mean()
        df[f'atr_ratio_{p}'] = df[f'atr_{p}'] / df['Close']
        df[f'natr_{p}'] = (df[f'atr_{p}'] / df['Close']) * 100

    # Statistical
    for p in [10,20,30,50]:
        df[f'std_{p}'] = df['Close'].rolling(p).std()
        df[f'skew_{p}'] = df['Close'].rolling(p).skew()
        df[f'kurt_{p}'] = df['Close'].rolling(p).kurt()
        df[f'median_{p}'] = df['Close'].rolling(p).median()
        df[f'quantile_25_{p}'] = df['Close'].rolling(p).quantile(0.25)
        df[f'quantile_75_{p}'] = df['Close'].rolling(p).quantile(0.75)
        df[f'iqr_{p}'] = df[f'quantile_75_{p}'] - df[f'quantile_25_{p}']

    # Lags
    for lag in [1,2,3,5,7,10,14,21,30]:
        df[f'close_lag_{lag}'] = df['Close'].shift(lag)
        df[f'volume_lag_{lag}'] = df['Volume'].shift(lag)
        df[f'high_lag_{lag}'] = df['High'].shift(lag)
        df[f'low_lag_{lag}'] = df['Low'].shift(lag)

    # Support/Resistance
    for p in [10,20,50,100]:
        df[f'high_max_{p}'] = df['High'].rolling(p).max()
        df[f'low_min_{p}'] = df['Low'].rolling(p).min()
        df[f'distance_high_{p}'] = (df['Close'] - df[f'high_max_{p}']) / df['Close']
        df[f'distance_low_{p}'] = (df['Close'] - df[f'low_min_{p}']) / df['Close']

    # Stochastic
    for p in [14,21]:
        low_min = df['Low'].rolling(p).min()
        high_max = df['High'].rolling(p).max()
        df[f'stoch_k_{p}'] = 100 * (df['Close'] - low_min) / (high_max - low_min)
        df[f'stoch_d_{p}'] = df[f'stoch_k_{p}'].rolling(3).mean()

    # Ichimoku
    high_9 = df['High'].rolling(9).max()
    low_9 = df['Low'].rolling(9).min()
    df['tenkan_sen'] = (high_9 + low_9) / 2
    high_26 = df['High'].rolling(26).max()
    low_26 = df['Low'].rolling(26).min()
    df['kijun_sen'] = (high_26 + low_26) / 2
    df['senkou_span_a'] = ((df['tenkan_sen'] + df['kijun_sen']) / 2).shift(26)
    high_52 = df['High'].rolling(52).max()
    low_52 = df['Low'].rolling(52).min()
    df['senkou_span_b'] = ((high_52 + low_52) / 2).shift(26)
    df['chikou_span'] = df['Close'].shift(-26)

    return df

# ============================================================================
# SEQUENCE CREATION
# ============================================================================

def create_sequences(data, target, lookback=60):
    """Create sequences for time-series prediction"""
    X, y = [], []
    for i in range(lookback, len(data)):
        X.append(data[i-lookback:i])
        y.append(target[i])
    return np.array(X), np.array(y)

# ============================================================================
# MODEL DEFINITIONS
# ============================================================================

def build_lstm(input_shape):
    """LSTM Model"""
    model = Sequential([
        layers.LSTM(50, return_sequences=True, input_shape=input_shape),
        layers.Dropout(0.2),
        layers.LSTM(50, return_sequences=False),
        layers.Dropout(0.2),
        layers.Dense(25, activation='relu'),
        layers.Dense(1)
    ])
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE), loss='mse')
    return model

def build_gru(input_shape):
    """GRU Model"""
    model = Sequential([
        layers.GRU(50, return_sequences=True, input_shape=input_shape),
        layers.Dropout(0.2),
        layers.GRU(50, return_sequences=False),
        layers.Dropout(0.2),
        layers.Dense(25, activation='relu'),
        layers.Dense(1)
    ])
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE), loss='mse')
    return model

def build_bilstm(input_shape):
    """Bidirectional LSTM Model"""
    model = Sequential([
        layers.Bidirectional(layers.LSTM(50, return_sequences=True), input_shape=input_shape),
        layers.Dropout(0.2),
        layers.Bidirectional(layers.LSTM(50, return_sequences=False)),
        layers.Dropout(0.2),
        layers.Dense(25, activation='relu'),
        layers.Dense(1)
    ])
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE), loss='mse')
    return model

def build_cnn_lstm(input_shape):
    """CNN-LSTM Hybrid Model"""
    model = Sequential([
        layers.Conv1D(64, 3, activation='relu', input_shape=input_shape),
        layers.MaxPooling1D(2),
        layers.Conv1D(32, 3, activation='relu'),
        layers.MaxPooling1D(2),
        layers.LSTM(50),
        layers.Dropout(0.2),
        layers.Dense(25, activation='relu'),
        layers.Dense(1)
    ])
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE), loss='mse')
    return model

def build_transformer(input_shape):
    """Transformer Model"""
    inputs = keras.Input(shape=input_shape)

    # Multi-head attention
    x = layers.MultiHeadAttention(num_heads=4, key_dim=input_shape[1])(inputs, inputs)
    x = layers.Dropout(0.2)(x)
    x = layers.LayerNormalization()(x + inputs)

    # Feed forward
    ff = layers.Dense(128, activation='relu')(x)
    ff = layers.Dense(input_shape[1])(ff)
    ff = layers.Dropout(0.2)(ff)
    x = layers.LayerNormalization()(x + ff)

    # Second attention layer
    x2 = layers.MultiHeadAttention(num_heads=4, key_dim=input_shape[1])(x, x)
    x2 = layers.Dropout(0.2)(x2)
    x2 = layers.LayerNormalization()(x2 + x)

    # Output
    x2 = layers.GlobalAveragePooling1D()(x2)
    x2 = layers.Dense(25, activation='relu')(x2)
    outputs = layers.Dense(1)(x2)

    model = keras.Model(inputs=inputs, outputs=outputs)
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE), loss='mse')
    return model

# ============================================================================
# TRAINING FUNCTION
# ============================================================================

def train_model(model, X_train, y_train, X_test, y_test, model_name):
    """Train a model with early stopping"""
    callbacks = [
        EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=0.00001)
    ]

    start_time = time.time()

    history = model.fit(
        X_train, y_train,
        validation_data=(X_test, y_test),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=callbacks,
        verbose=0
    )

    training_time = time.time() - start_time

    # Predictions
    y_pred = model.predict(X_test, verbose=0)

    # Metrics
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))

    print(f"      ✅ {model_name:15s} R²={r2:.4f} | MAE={mae:.2f} | Time={training_time:.1f}s")

    return {
        'model': model_name,
        'r2': r2,
        'mae': mae,
        'rmse': rmse,
        'training_time': training_time,
        'epochs_completed': len(history.history['loss']),
        'history': history.history
    }, y_pred

# ============================================================================
# MAIN PROCESSING LOOP
# ============================================================================

def process_stock(ticker, stock_num, total_stocks):
    """Process one stock with all 5 DL models"""
    print(f"\n{'='*80}")
    print(f"Stock {stock_num}/{total_stocks}: {ticker}")
    print(f"{'='*80}")

    try:
        # Load data
        print("   📥 Loading data...")
        df = load_csv(ticker)
        print(f"      ✅ {len(df)} rows loaded")

        # Generate features
        print("   🔧 Generating 325 features...")
        df = generate_325_features(df)
        df = df.dropna()
        print(f"      ✅ Features generated")

        # Prepare data
        print("   📊 Creating sequences...")
        exclude = ['Open', 'High', 'Low', 'Close', 'Volume']
        feature_cols = [c for c in df.columns if c not in exclude]

        # Features and target
        X_data = df[feature_cols].values
        y_data = df['Close'].values

        # Normalize
        scaler_X = MinMaxScaler()
        scaler_y = MinMaxScaler()
        X_scaled = scaler_X.fit_transform(X_data)
        y_scaled = scaler_y.fit_transform(y_data.reshape(-1, 1)).flatten()

        # Create sequences
        X, y = create_sequences(X_scaled, y_scaled, lookback=LOOKBACK)
        print(f"      ✅ {len(X)} sequences created")

        # Train/test split
        split_idx = int(0.8 * len(X))
        X_train, X_test = X[:split_idx], X[split_idx:]
        y_train, y_test = y[:split_idx], y[split_idx:]
        print(f"      ✅ Train: {len(X_train)} | Test: {len(X_test)}")

        # Input shape
        input_shape = (X_train.shape[1], X_train.shape[2])

        # Train all models
        print(f"\n   🤖 Training 5 Deep Learning models...")
        results = {}
        predictions = {}

        models_to_train = [
            ('LSTM', build_lstm),
            ('GRU', build_gru),
            ('BiLSTM', build_bilstm),
            ('CNN_LSTM', build_cnn_lstm),
            ('Transformer', build_transformer)
        ]

        for model_name, build_fn in models_to_train:
            try:
                # Build model
                model = build_fn(input_shape)

                # Train
                result, y_pred = train_model(model, X_train, y_train, X_test, y_test, model_name)

                # Store results
                results[model_name] = result
                predictions[model_name] = y_pred

                # Clear memory
                del model
                keras.backend.clear_session()
                gc.collect()

            except Exception as e:
                print(f"      ❌ {model_name} failed: {str(e)}")
                continue

        # Save results
        print(f"\n   💾 Saving results...")
        output_dir = Path('/content/drive/MyDrive/MarketLab_BEAST/results_day2')
        output_dir.mkdir(exist_ok=True, parents=True)

        clean_ticker = ticker.replace('.NS', '').replace('&', 'AND').replace('-', '_')

        # Save metrics
        metrics_df = pd.DataFrame([
            {
                'model': r['model'],
                'r2': r['r2'],
                'mae': r['mae'],
                'rmse': r['rmse'],
                'training_time': r['training_time'],
                'epochs_completed': r['epochs_completed']
            }
            for r in results.values()
        ])
        metrics_df.to_csv(output_dir / f"{clean_ticker}_dl_results.csv", index=False)

        # Save training history
        for model_name, result in results.items():
            history_file = output_dir / f"{clean_ticker}_{model_name}_history.json"
            with open(history_file, 'w') as f:
                json.dump(result['history'], f)

        # Find best model
        best_model = max(results.values(), key=lambda x: x['r2'])
        print(f"   ✅ DONE! Best: {best_model['model']} (R²={best_model['r2']:.4f})")

        return {
            'ticker': ticker,
            'best_model': best_model['model'],
            'best_r2': best_model['r2'],
            'avg_r2': np.mean([r['r2'] for r in results.values()]),
            'num_models': len(results),
            'num_features': len(feature_cols)
        }

    except Exception as e:
        print(f"   ❌ ERROR: {str(e)}")
        import traceback
        traceback.print_exc()
        return None

# ============================================================================
# EXECUTE
# ============================================================================

print("🚀 Starting Day 2 Training...\n")
start_time = time.time()

summary_results = []
for i, stock in enumerate(STOCKS, 1):
    result = process_stock(stock, i, len(STOCKS))
    if result:
        summary_results.append(result)

# Save summary
if summary_results:
    summary_df = pd.DataFrame(summary_results)
    output_dir = Path('/content/drive/MyDrive/MarketLab_BEAST/results_day2')
    summary_df.to_csv(output_dir / 'day2_summary.csv', index=False)

    print("\n" + "="*80)
    print("🎉 DAY 2 COMPLETE!")
    print("="*80)
    print(f"✅ Processed: {len(summary_results)}/{len(STOCKS)} stocks")
    print(f"✅ Models trained: {len(summary_results) * 5} total")
    print(f"✅ Average R²: {summary_df['avg_r2'].mean():.4f}")
    print(f"✅ Best R²: {summary_df['best_r2'].max():.4f}")
    print(f"⏱️  Total time: {(time.time() - start_time)/60:.1f} minutes")
    print("="*80)
    print("\n📂 Results saved in: /MarketLab_BEAST/results_day2/")
    print("🎉 Ready for Day 3 - SHAP Analysis!")

else:
    print("\n❌ No stocks completed successfully. Check errors above.")

🔥 MARKETLAB DAY 2 - DEEP LEARNING
TensorFlow: 2.19.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

🚀 Starting Day 2 Training...


Stock 1/10: RELIANCE.NS
   📥 Loading data...
      ✅ 5193 rows loaded
   🔧 Generating 325 features...
      ✅ Features generated
   📊 Creating sequences...
      ✅ 4848 sequences created
      ✅ Train: 3878 | Test: 970

   🤖 Training 5 Deep Learning models...
      ✅ LSTM            R²=0.2148 | MAE=0.08 | Time=31.8s
      ✅ GRU             R²=0.5415 | MAE=0.06 | Time=29.9s
      ✅ BiLSTM          R²=0.2285 | MAE=0.08 | Time=30.1s
      ✅ CNN_LSTM        R²=0.4133 | MAE=0.07 | Time=15.6s
      ✅ Transformer     R²=0.6853 | MAE=0.05 | Time=70.2s

   💾 Saving results...
   ✅ DONE! Best: Transformer (R²=0.6853)

Stock 2/10: TCS.NS
   📥 Loading data...
      ✅ 5194 rows loaded
   🔧 Generating 325 features...
      ✅ Features generated
   📊 Creating sequences...
      ✅ 4849 sequences created
      ✅ Train: 3879 | Test: 970

   🤖 Trainin

Traceback (most recent call last):
  File "/tmp/ipykernel_31424/4225738484.py", line 387, in process_stock
    X_scaled = scaler_X.fit_transform(X_data)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/sklearn/utils/_set_output.py", line 319, in wrapped
    data_to_wrap = f(self, X, *args, **kwargs)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/sklearn/base.py", line 918, in fit_transform
    return self.fit(X, **fit_params).transform(X)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_data.py", line 447, in fit
    return self.partial_fit(X, y)
           ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/sklearn/base.py", line 1389, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_da

      ✅ 5195 rows loaded
   🔧 Generating 325 features...
      ✅ Features generated
   📊 Creating sequences...
      ✅ 4850 sequences created
      ✅ Train: 3880 | Test: 970

   🤖 Training 5 Deep Learning models...
      ✅ LSTM            R²=-1.7696 | MAE=0.15 | Time=20.9s
      ✅ GRU             R²=-0.7063 | MAE=0.12 | Time=20.7s
      ✅ BiLSTM          R²=-0.5990 | MAE=0.11 | Time=30.6s
      ✅ CNN_LSTM        R²=0.6713 | MAE=0.05 | Time=18.3s
      ✅ Transformer     R²=0.0945 | MAE=0.07 | Time=81.5s

   💾 Saving results...
   ✅ DONE! Best: CNN_LSTM (R²=0.6713)

Stock 5/10: TATAMOTORS.NS
   📥 Loading data...
   ❌ ERROR: [Errno 2] No such file or directory: '/content/drive/MyDrive/MarketLab_BEAST/stock_data/TATAMOTORS.NS.csv'

Stock 6/10: BAJFINANCE.NS
   📥 Loading data...


Traceback (most recent call last):
  File "/tmp/ipykernel_31424/4225738484.py", line 366, in process_stock
    df = load_csv(ticker)
         ^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_31424/4225738484.py", line 49, in load_csv
    with open(path, 'r') as f:
         ^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/MarketLab_BEAST/stock_data/TATAMOTORS.NS.csv'


      ✅ 5193 rows loaded
   🔧 Generating 325 features...
      ✅ Features generated
   📊 Creating sequences...
      ✅ 4847 sequences created
      ✅ Train: 3877 | Test: 970

   🤖 Training 5 Deep Learning models...
      ✅ LSTM            R²=-3.3646 | MAE=0.19 | Time=34.8s
      ✅ GRU             R²=-0.4821 | MAE=0.11 | Time=23.1s
      ✅ BiLSTM          R²=-2.3114 | MAE=0.16 | Time=45.7s
      ✅ CNN_LSTM        R²=-3.0349 | MAE=0.18 | Time=14.7s
      ✅ Transformer     R²=0.4965 | MAE=0.05 | Time=80.1s

   💾 Saving results...
   ✅ DONE! Best: Transformer (R²=0.4965)

Stock 7/10: ASIANPAINT.NS
   📥 Loading data...
      ✅ 5193 rows loaded
   🔧 Generating 325 features...
      ✅ Features generated
   📊 Creating sequences...
      ✅ 4848 sequences created
      ✅ Train: 3878 | Test: 970

   🤖 Training 5 Deep Learning models...
      ✅ LSTM            R²=-0.8285 | MAE=0.09 | Time=36.7s
      ✅ GRU             R²=-1.1645 | MAE=0.10 | Time=54.8s
      ✅ BiLSTM          R²=-1.6101 | MAE=0.11

Traceback (most recent call last):
  File "/tmp/ipykernel_31424/4225738484.py", line 387, in process_stock
    X_scaled = scaler_X.fit_transform(X_data)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/sklearn/utils/_set_output.py", line 319, in wrapped
    data_to_wrap = f(self, X, *args, **kwargs)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/sklearn/base.py", line 918, in fit_transform
    return self.fit(X, **fit_params).transform(X)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_data.py", line 447, in fit
    return self.partial_fit(X, y)
           ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/sklearn/base.py", line 1389, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_da

      ✅ 3488 rows loaded
   🔧 Generating 325 features...
      ✅ Features generated
   📊 Creating sequences...
      ✅ 3148 sequences created
      ✅ Train: 2518 | Test: 630

   🤖 Training 5 Deep Learning models...
      ✅ LSTM            R²=0.0533 | MAE=0.18 | Time=24.7s
      ✅ GRU             R²=0.6639 | MAE=0.12 | Time=13.4s
      ✅ BiLSTM          R²=-0.1142 | MAE=0.20 | Time=23.1s
      ✅ CNN_LSTM        R²=0.5424 | MAE=0.14 | Time=12.2s
      ✅ Transformer     R²=-1.3000 | MAE=0.32 | Time=69.2s

   💾 Saving results...
   ✅ DONE! Best: GRU (R²=0.6639)

🎉 DAY 2 COMPLETE!
✅ Processed: 7/10 stocks
✅ Models trained: 35 total
✅ Average R²: -0.3443
✅ Best R²: 0.8931
⏱️  Total time: 24.4 minutes

📂 Results saved in: /MarketLab_BEAST/results_day2/
🎉 Ready for Day 3 - SHAP Analysis!
